In [ ]:
"""[Kaggle L4] UMBRAL POR POSICION en el DFS + anatomia por celda de la NLL (4B).

Hallazgo que motiva esto (22-ago, datos del kernel tier1-4b): con el umbral FIJO de 1,609,
37/40 verdades de eval quedan fuera del pool del 4B, y 9 de ellas con confianza por celda
>= 98% -- el caso extremo es 135a2760: 870 celdas al 99,8% cada una, excluida por 0,04 nats.
La suma con presupuesto fijo es un impuesto al tamano: L=30 entra con 94,8%/celda, L=300
exige 99,5%/celda. Eso muerde el factor GT-en-pool (0,42), la palanca x2 del score.

Este kernel mide DOS cosas sobre las MISMAS 40 tareas de eval (poblacion no memorizada):

1. ESTRUCTURAL (exacto, sin ruido de busqueda): del vector de NLL por celda de la GT, si el
   camino de la GT cabe bajo tau(pos) = max(1.609, b*pos) para b en {0.015, 0.0202, 0.025,
   0.03}, y bajo la regla vieja (cumsum < 1.609 en todo prefijo). Ademas anatomia: max por
   celda y fraccion de la NLL total en las 5 peores celdas (si se concentra, una perdida
   ponderada por celdas-dificiles seria el siguiente experimento de ENTRENAMIENTO).

2. DE BUSQUEDA (el DFS real): pools, tiempos y aciertos con tau(pos) = max(1.609,
   0.0202*pos) (98%/celda; identico al actual bajo L=80) contra el baseline ya medido
   (3/40, 397s, kernel tier1-4b).

Prediccion pre-registrada: la regla nueva admite estructuralmente la GT en 8-12/40 (vs 3);
el coste de decode sube pero acotado (98%/celda sigue siendo estricto); el riesgo a medir
es basura admitida en rejillas grandes.
"""
import glob
import json
import os
import time

import numpy as np
import torch
from torch.nn.attention import SDPBackend, sdpa_kernel
from transformers import AutoModelForCausalLM

T00 = time.time()
_SDPA = [SDPBackend.EFFICIENT_ATTENTION, SDPBackend.FLASH_ATTENTION, SDPBackend.MATH]
IM_START, USER, ASSIST, NL, EOS = 14, 11, 12, 10, 15
ARC_TOKENS = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15]
_TOKID = torch.tensor(ARC_TOKENS, dtype=torch.long, device="cuda:0")
THR0 = -float(np.log(0.2))          # 1.609, el umbral fijo de produccion
B_RATE = 0.0202                     # 98%/celda; coincide con 1.609/80 -> identico bajo L=80
BS = (0.015, 0.0202, 0.025, 0.03)   # barrido estructural, gratis


def tau(pos):
    return max(THR0, B_RATE * pos)


_COMP = None
for c in glob.glob("/kaggle/input/**/arc-agi_evaluation_challenges.json", recursive=True):
    _COMP = os.path.dirname(c)
    break
assert _COMP, "no se encontro la data de la competencia"
MDIR = None
for c in glob.glob("/kaggle/input/**/config.json", recursive=True):
    d = os.path.dirname(c)
    if glob.glob(os.path.join(d, "*.safetensors")):
        MDIR = d
        break
assert MDIR, "no se encontro el modelo adjunto"
print(f"[setup] comp={_COMP} modelo={MDIR}", flush=True)

CHAL = json.load(open(f"{_COMP}/arc-agi_evaluation_challenges.json"))
SOLS = json.load(open(f"{_COMP}/arc-agi_evaluation_solutions.json"))

KEYS = ["e8686506", "20270e3b", "7b5033c1", "b0039139", "78332cb0", "28a6681f", "d35bdbdc",
        "dbff022c", "4c3d4a41", "446ef5d2", "dd6b8c4b", "3dc255db", "16de56c4", "97d7923e",
        "2b83f449", "bf45cf4b", "faa9f03d", "eee78d87", "31f7f899", "5dbc8537", "291dc1e1",
        "f931b4a8", "f560132c", "271d71e2", "247ef758", "7666fa5d", "21897d95", "1818057f",
        "5545f144", "135a2760", "1ae2feb7", "38007db0", "3a25b0d8", "581f7754", "6e453dd6",
        "65b59efc", "b9e38dc0", "53fb4810", "7ed72f31", "c7f57c3e"]
GATE = "38007db0"


def grid_tokens(g):
    out = []
    for i, row in enumerate(g):
        out.extend(int(c) for c in row)
        if i < len(g) - 1:
            out.append(NL)
    return out


def pair_tokens(inp, out):
    return [IM_START, USER, NL] + grid_tokens(inp) + [EOS] + \
           [IM_START, ASSIST, NL] + grid_tokens(out) + [EOS]


def query_tokens(inp):
    return [IM_START, USER, NL] + grid_tokens(inp) + [EOS] + [IM_START, ASSIST, NL]


MODEL = AutoModelForCausalLM.from_pretrained(MDIR, torch_dtype=torch.bfloat16,
                                             device_map="cuda:0", attn_implementation="sdpa")
MODEL.eval()
MODEL.config.use_cache = True
print(f"[setup] 4B bf16 | mem={torch.cuda.memory_allocated()/1e9:.1f}GB "
      f"({time.time()-T00:.0f}s)", flush=True)


@torch.no_grad()
def prefill(toks, chunk=512):
    cache, out = None, None
    for i in range(0, len(toks), chunk):
        with sdpa_kernel(_SDPA):
            out = MODEL(input_ids=torch.tensor([toks[i:i + chunk]], device="cuda:0",
                                               dtype=torch.long),
                        past_key_values=cache, use_cache=True, return_dict=True)
        cache = out.past_key_values
    return out.logits[:, -1], cache


@torch.no_grad()
def nll_vector(prompt, ans):
    """NLL POR TOKEN de la respuesta (no la suma): la materia prima de todo el analisis."""
    last, cache = prefill(prompt)
    parts = [last.float()]
    if len(ans) > 1:
        with sdpa_kernel(_SDPA):
            o = MODEL(input_ids=torch.tensor([ans[:-1]], device="cuda:0", dtype=torch.long),
                      past_key_values=cache, use_cache=True, return_dict=True)
        parts.append(o.logits[0].float())
    lg = torch.cat([x.view(-1, x.shape[-1]) for x in parts], 0)
    v = -torch.log_softmax(lg, -1).gather(
        1, torch.tensor(ans, device="cuda:0").unsqueeze(1)).squeeze(1)
    out = [round(x, 5) for x in v.tolist()]
    del parts, lg, cache
    torch.cuda.empty_cache()
    return out


def toks_to_grid(toks):
    rows, row = [], []
    for t in toks:
        if t == NL:
            rows.append(row)
            row = []
        elif 0 <= t <= 9:
            row.append(t)
        else:
            return None
    if row:
        rows.append(row)
    if not rows or not rows[0] or any(len(r) != len(rows[0]) for r in rows):
        return None
    return rows


@torch.no_grad()
def dfs_budget(prompt, depth, budget=60):
    """El DFS de produccion con UNA diferencia: el umbral crece con la posicion.
    La poda sigue siendo monotona (el score acumulado solo sube; la cota solo depende de
    la posicion), asi que la correccion del algoritmo no cambia."""
    t0 = time.time()
    last, cache0 = prefill(prompt)
    leaves, path = [], []

    def rec(logits, score, cache, left):
        if time.time() - t0 > budget or left <= 1:
            return
        lf = logits.float()
        nll = (score + torch.logsumexp(lf, dim=-1, keepdim=True)
               - lf.index_select(-1, _TOKID)).cpu()
        lim = tau(len(path) + 1)
        cands = []
        for j, t in enumerate(ARC_TOKENS):
            sc = nll[0, j].item()
            if sc < lim:
                if t == EOS:
                    leaves.append((sc, list(path)))
                else:
                    cands.append((sc, t))
        for sc, t in sorted(cands):
            if time.time() - t0 > budget:
                return
            with sdpa_kernel(_SDPA):
                o = MODEL(input_ids=torch.tensor([[t]], device="cuda:0", dtype=torch.long),
                          past_key_values=cache, use_cache=True, return_dict=True)
            path.append(t)
            rec(o.logits[:, -1], sc, o.past_key_values, left - 1)
            path.pop()

    rec(last, 0.0, cache0, depth)
    return leaves, time.time() - t0


def make_prompt(k):
    t = CHAL[k]
    p = []
    for x in t["train"]:
        p += pair_tokens(x["input"], x["output"])
    p += query_tokens(t["test"][0]["input"])
    return p


_v = nll_vector(make_prompt(GATE), grid_tokens(SOLS[GATE][0]) + [EOS])
print(f"[gate] {GATE}: NLL de la verdad {sum(_v):.4f}", flush=True)
assert sum(_v) < 1.0, f"gate FALLA ({sum(_v):.3f}): modelo mal cargado, se aborta"

RES = {"tareas": [], "b_rate": B_RATE, "presupuesto_s": 60}
for i, k in enumerate(KEYS):
    prompt = make_prompt(k)
    gt = SOLS[k][0]
    gt_toks = grid_tokens(gt) + [EOS]

    # 1) ESTRUCTURAL: el vector por celda de la GT decide admisibilidad sin buscar
    v = nll_vector(prompt, gt_toks)
    cum = np.cumsum(v)
    adm_vieja = bool(cum[-1] < THR0) and bool(np.all(cum < THR0))
    adm = {}
    for b in BS:
        lims = np.maximum(THR0, b * np.arange(1, len(v) + 1))
        adm[str(b)] = bool(np.all(cum < lims))
    top5 = sorted(v, reverse=True)[:5]
    anat = {"nll_total": round(float(cum[-1]), 4), "max_celda": round(max(v), 4),
            "top5_frac": round(sum(top5) / max(cum[-1], 1e-9), 3),
            "n_celdas_50pct": int(np.searchsorted(np.cumsum(sorted(v, reverse=True)),
                                                  0.5 * cum[-1]) + 1)}

    # 2) BUSQUEDA con la regla nueva
    try:
        hojas, secs = dfs_budget(prompt, min(931, len(gt_toks) + 40))
    except torch.cuda.OutOfMemoryError:
        print(f"[{i+1}/{len(KEYS)}] {k}: OOM en busqueda", flush=True)
        torch.cuda.empty_cache()
        hojas, secs = [], -1.0
    hojas.sort(key=lambda x: x[0])
    grids = [(sc, toks_to_grid(p)) for sc, p in hojas]
    best_ok = bool(grids and grids[0][1] == gt)
    dos_ok = bool(any(g == gt for _, g in grids[:2]))
    alguno = bool(any(g == gt for _, g in grids))

    RES["tareas"].append({
        "task": k, "L": len(gt_toks), "anatomia": anat,
        "admisible_vieja": adm_vieja, "admisible_nueva": adm,
        "pool": len(grids), "best_ok": best_ok, "pass2_ok": dos_ok, "gt_en_pool": alguno,
        "best_score": round(grids[0][0], 4) if grids else None,
        "decode_s": round(secs, 1)})
    print(f"[{i+1}/{len(KEYS)}] {k}: L={len(gt_toks)} adm_vieja={adm_vieja} "
          f"adm_b0202={adm['0.0202']} | pool {len(grids)} | gt_en_pool {alguno} | "
          f"pass2 {dos_ok} | top5_frac {anat['top5_frac']} | {secs:.0f}s", flush=True)
    json.dump(RES, open("/kaggle/working/dfs_budget.json", "w"), indent=1)
    torch.cuda.empty_cache()

t = RES["tareas"]
resumen = {
    "n": len(t),
    "admisible_vieja": sum(x["admisible_vieja"] for x in t),
    "admisible_nueva": {b: sum(x["admisible_nueva"][b] for x in t) for b in map(str, BS)},
    "gt_en_pool_busqueda": sum(x["gt_en_pool"] for x in t),
    "pass2": sum(x["pass2_ok"] for x in t),
    "pool_total": sum(x["pool"] for x in t),
    "decode_total_s": round(sum(max(x["decode_s"], 0) for x in t), 1),
    "baseline_tier1_4b": {"gt_en_pool": 3, "pass2": 3, "decode_s": 397},
    "anatomia_mediana_top5_frac": round(float(np.median([x["anatomia"]["top5_frac"]
                                                         for x in t])), 3),
}
RES["resumen"] = resumen
json.dump(RES, open("/kaggle/working/dfs_budget.json", "w"), indent=1)
print("RESULT_JSON:" + json.dumps(resumen), flush=True)
print(f"[fin] {time.time()-T00:.0f}s", flush=True)
